# DistilBERT for PII Detection

This notebook documents the DistilBERT token classification experiment for PII detection.

## 1. Setup

Run the next cell if the environment has not been installed yet.

In [ ]:
# Uncomment if dependencies are not installed yet
# !pip install -r requirements.txt
# !pip install -r models/transformer/requirements.txt

import os
from pathlib import Path
import json

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / 'models').exists() else CURRENT_DIR.parent
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
print('Notebook working directory:', Path.cwd().resolve())

In [ ]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2. Check the processed dataset

In [ ]:
for rel in [
    'data/processed/train.json',
    'data/processed/val.json',
    'data/processed/test_internal.json',
]:
    path = PROJECT_ROOT / rel
    print(rel, 'exists =', path.exists())

## 3. Train DistilBERT

This cell reproduces the baseline DistilBERT configuration used in the repository.

In [ ]:
!python models/transformer/train_transformer.py \
  --model-key distilbert \
  --artifact-prefix distilbert_recovered \
  --output-dir models/transformer/distilbert-pii \
  --epochs 3 \
  --batch-size 16 \
  --learning-rate 5e-5 \
  --max-length 256 \
  --chunk-size 128 \
  --gradient-accumulation-steps 1 \
  --loss-mode sqrt_balanced

## 4. View the training metrics

In [ ]:
metrics_path = PROJECT_ROOT / 'results/metrics/distilbert_recovered_metrics.json'
with metrics_path.open('r', encoding='utf-8') as f:
    metrics = json.load(f)

print(json.dumps({
    'token_f1': metrics['token_level']['f1'],
    'entity_f1': metrics['entity_level']['f1'],
    'total_tokens': metrics['total_tokens'],
}, indent=2))

## 5. Re-evaluate the saved model

This is optional if you want to regenerate the metrics and prediction CSV from the saved model.

In [ ]:
!python models/transformer/evaluate_saved_transformer.py \
  --model-key distilbert \
  --model-dir models/transformer/distilbert-pii \
  --artifact-prefix distilbert_recheck

## 6. Run manual inference

In [ ]:
!python models/transformer/predict.py \
  "My name is John Smith, my email is john@example.com, and my website is https://john.dev" \
  --model-key distilbert \
  --min-confidence 0.5